# BIO Dataset Split — species-stratified train/val/test

Splits `../animalia_dataset/all_species_bio.jsonl` (the combined per-sentence
BIO file, 7,281 species) into `train.jsonl`, `val.jsonl`, `test.jsonl`.

The split is done **per species** (using the `source` field written into each
record), never per sentence — a species never appears in more than one split,
avoiding phrasing leakage between train and test.

Because entity types are extremely imbalanced (`T` ≈ 32% of all instances vs.
`Wwx` ≈ 0.5%), a plain random species split can accidentally starve a rare
entity type from val/test. This notebook instead uses a greedy **iterative
stratification** assignment: species containing the globally rarest entity
types are assigned first, each going to whichever split currently has the
largest deficit relative to its 80/10/10 target for that species' entity mix.

Output: `../bio_dataset_split/{train,val,test}.jsonl`, same
`{"tokens": [...], "biotags": [...], "source": ...}` line shape as the input.

## Cell 0 – Imports and configuration

In [7]:
import json
from collections import Counter
from pathlib import Path

COMBINED_PATH = Path("../animalia_dataset/all_species_bio.jsonl")
OUT_DIR = Path("../animalia_dataset/bio_dataset_split")
OUT_DIR.mkdir(exist_ok=True)

TARGET_RATIO = {"train": 0.8, "val": 0.1, "test": 0.1}
assert abs(sum(TARGET_RATIO.values()) - 1.0) < 1e-9

print("Reading from:", COMBINED_PATH.resolve())
print("Writing to  :", OUT_DIR.resolve())

Reading from: /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_annotation/animalia_dataset/all_species_bio.jsonl
Writing to  : /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_annotation/animalia_dataset/bio_dataset_split


## Cell 1 – Load combined file & group records by species

In [2]:
species_records = {}   # source filename -> list of record dicts
species_entity_counts = {}   # source filename -> Counter(entity_type -> count)

with COMBINED_PATH.open(encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        src = rec["source"]
        species_records.setdefault(src, []).append(rec)

        counts = species_entity_counts.setdefault(src, Counter())
        for tag in rec["biotags"]:
            if tag.startswith("B-") or tag.startswith("U-"):
                counts[tag[2:]] += 1
            elif tag == "O":
                continue
            # (this file only ever contains B-/I-/O, kept U- check for safety
            #  in case this notebook is ever pointed at a BILOU-style file)

n_species = len(species_records)
n_sentences = sum(len(v) for v in species_records.values())
print(f"Species loaded   : {n_species:,}")
print(f"Sentences loaded : {n_sentences:,}")

Species loaded   : 7,281
Sentences loaded : 191,337


## Cell 2 – Global entity-type counts & species processing order

In [3]:
global_counts = Counter()
for counts in species_entity_counts.values():
    global_counts.update(counts)

print("Global entity-type counts (rarest first):")
for et, c in sorted(global_counts.items(), key=lambda x: x[1]):
    print(f"  {et:8s} {c:,}")

# Process species that contain a globally rare entity type first — they have
# the least flexibility for where they can land without skewing that type.
def species_priority(sp):
    counts = species_entity_counts[sp]
    if not counts:
        return float("inf")  # no whitelisted entities at all - process last, size-balance only
    return min(global_counts[et] for et in counts)

species_order = sorted(species_records.keys(), key=species_priority)
print(f"\nFirst 5 species in priority order: {species_order[:5]}")

Global entity-type counts (rarest first):
  Wwx      626
  GSI      774
  Ww0      1,204
  JOi      1,383
  tg       1,527
  Lb       1,585
  tR       2,347
  Ri       2,507
  Wwp      3,301
  tx       3,535
  ab       5,005
  tp       5,268
  N_R_i    5,571
  F_R_i    5,576
  Lp       6,052
  Li       6,772
  Wwb      7,461
  am       9,757
  Wwi      10,441
  T        38,835

First 5 species in priority order: ['0072_Acinonyx_jubatus_bio.json', '0088_Acomys_cahirinus_bio.json', '0092_Aconaemys_fuscus_bio.json', '0097_Acrobates_pygmaeus_bio.json', '0131_Aepyprymnus_rufescens_bio.json']


## Cell 3 – Iterative stratified assignment (per-label budget, rarest label first)

Standard iterative stratification (Sechidis et al. 2011), adapted to
species-level assignment. **Why not a per-species deficit sum:** almost
every species contains `T` (present in ~32% of all instances, with a huge
absolute train budget), so summing deficits across a species' entity types
lets `T` swamp the decision — a species only prioritized for its rare entity
would still get routed by `T`'s pull, and rare entities could end up 100% in
`train` with nothing left for val/test.

Instead: process **entity types** from rarest to most common. For each
type, assign its remaining candidate species one at a time to whichever
split currently has the largest **remaining budget for that specific type**
(`TARGET_RATIO[split] * global_count[type]`, decremented as species are
assigned) — tie-broken by overall split-size deficit. This way a common
entity's huge budget never overrides a rare entity's placement decision.
Species with no whitelisted entities at all are assigned last, purely by
split-size balance.

In [4]:
remaining = {
    k: {et: TARGET_RATIO[k] * c for et, c in global_counts.items()}
    for k in TARGET_RATIO
}
assigned_counts = {k: Counter() for k in TARGET_RATIO}
assigned_n_species = {k: 0 for k in TARGET_RATIO}
split_of = {}
unassigned = set(species_records.keys())
total_species = len(species_records)

entity_order = sorted(global_counts, key=lambda et: global_counts[et])

for et in entity_order:
    candidates = sorted(
        (sp for sp in unassigned if et in species_entity_counts[sp]),
        key=species_priority,
    )
    for sp in candidates:
        best_split = max(
            TARGET_RATIO,
            key=lambda k: (remaining[k][et], TARGET_RATIO[k] * total_species - assigned_n_species[k]),
        )
        split_of[sp] = best_split
        for et2, c in species_entity_counts[sp].items():
            assigned_counts[best_split][et2] += c
            remaining[best_split][et2] -= c
        assigned_n_species[best_split] += 1
        unassigned.discard(sp)

# species with no whitelisted entities at all: assign by pure size balance
for sp in sorted(unassigned):
    best_split = max(TARGET_RATIO, key=lambda k: TARGET_RATIO[k] * total_species - assigned_n_species[k])
    split_of[sp] = best_split
    assigned_n_species[best_split] += 1

print("Species per split:")
for k in TARGET_RATIO:
    print(f"  {k:6s} {assigned_n_species[k]:,} species")

Species per split:
  train  5,807 species
  val    736 species
  test   738 species


## Cell 4 – Write train/val/test JSONL files

In [5]:
sentence_counts = {k: 0 for k in TARGET_RATIO}
file_handles = {k: (OUT_DIR / f"{k}.jsonl").open("w", encoding="utf-8") for k in TARGET_RATIO}

try:
    for sp, records in species_records.items():
        k = split_of[sp]
        for rec in records:
            file_handles[k].write(json.dumps(rec, ensure_ascii=False) + "\n")
            sentence_counts[k] += 1
finally:
    for fh in file_handles.values():
        fh.close()

print("Sentences per split:")
for k in TARGET_RATIO:
    path = OUT_DIR / f"{k}.jsonl"
    print(f"  {k:6s} {sentence_counts[k]:,} sentences -> {path.resolve()}")

Sentences per split:
  train  152,632 sentences -> /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_annotation/animalia_dataset/bio_dataset_split/train.jsonl
  val    19,337 sentences -> /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_annotation/animalia_dataset/bio_dataset_split/val.jsonl
  test   19,368 sentences -> /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_annotation/animalia_dataset/bio_dataset_split/test.jsonl


## Cell 5 – Validation: entity-type coverage per split

In [6]:
print(f"{'entity':8s} {'global':>8s}  " + "  ".join(f"{k:>16s}" for k in TARGET_RATIO))
for et in sorted(global_counts, key=lambda e: -global_counts[e]):
    g = global_counts[et]
    row = []
    for k in TARGET_RATIO:
        c = assigned_counts[k][et]
        pct_of_type = 100 * c / g if g else 0.0
        row.append(f"{c:>7,} ({pct_of_type:4.1f}%)")
    print(f"{et:8s} {g:>8,}  " + "  ".join(f"{r:>16s}" for r in row))

print("\nFlagging entity types with zero instances in any split:")
for et in global_counts:
    for k in TARGET_RATIO:
        c = assigned_counts[k][et]
        if c == 0:
            print(f"  WARNING: entity '{et}' has ZERO instances in split '{k}'")

entity     global             train               val              test
T          38,835    31,051 (80.0%)     3,920 (10.1%)     3,864 ( 9.9%)
Wwi        10,441     8,335 (79.8%)     1,031 ( 9.9%)     1,075 (10.3%)
am          9,757     7,793 (79.9%)       980 (10.0%)       984 (10.1%)
Wwb         7,461     5,938 (79.6%)       748 (10.0%)       775 (10.4%)
Li          6,772     5,368 (79.3%)       688 (10.2%)       716 (10.6%)
Lp          6,052     4,811 (79.5%)       611 (10.1%)       630 (10.4%)
F_R_i       5,576     4,448 (79.8%)       558 (10.0%)       570 (10.2%)
N_R_i       5,571     4,454 (79.9%)       561 (10.1%)       556 (10.0%)
tp          5,268     4,215 (80.0%)       527 (10.0%)       526 (10.0%)
ab          5,005     4,003 (80.0%)       502 (10.0%)       500 (10.0%)
tx          3,535     2,852 (80.7%)       342 ( 9.7%)       341 ( 9.6%)
Wwp         3,301     2,641 (80.0%)       329 (10.0%)       331 (10.0%)
Ri          2,507     2,005 (80.0%)       251 (10.0%)       251 